# 03 · Daily observations: completeness, value sanity, seasonality, consistency

The daily table (~10M rows, °F and inches) is the source the annual indices are computed from.
`load_daily` reads it compactly (categorical ID, float32 values: ~225 MB in memory).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from proto_star.climate import MAX_MISSING_DAYS, METRICS, annual_to_long
from proto_star.data.protostar import (
    REGIONS,
    city_table,
    load_annual,
    load_city_info,
    load_daily,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
REGION_PALETTE = dict(zip(REGIONS, sns.color_palette("tab10", len(REGIONS)), strict=True))

In [ ]:
daily = load_daily()
annual = load_annual()
cities = city_table(annual, load_city_info())
daily = daily.merge(
    cities[["ID", "Name", "region"]].astype({"ID": daily["ID"].dtype}), on="ID", how="left"
)
daily["Year"] = daily["Date"].dt.year
daily["Month"] = daily["Date"].dt.month
print(f"{len(daily):,} rows, {daily.memory_usage(deep=True).sum() / 1e6:.0f} MB")
daily.head()

## 1. Missing values over time

In [ ]:
miss_by_year = daily.groupby("Year")[["tmax", "tmin", "prcp"]].apply(lambda g: g.isna().mean())
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
miss_by_year.plot(ax=axes[0])
axes[0].set_title("Share of daily values missing, by year (all cities)")

miss_city = (
    daily[daily["Year"] >= 1989]
    .groupby("Name", observed=True)[["tmax", "tmin", "prcp"]]
    .apply(lambda g: g.isna().mean())
)
sns.histplot(miss_city, bins=40, ax=axes[1], element="step", fill=False)
axes[1].set_title("Per-city missing share, 1989-2018")
axes[1].set_yscale("log")
plt.tight_layout()
miss_city.sort_values("prcp", ascending=False).head(10).round(3)

In [ ]:
# Do tmax/tmin go missing together?
both = pd.crosstab(daily["tmax"].isna(), daily["tmin"].isna(), normalize="all").round(4)
both.index.name, both.columns.name = "tmax missing", "tmin missing"
both

## 2. Value sanity checks

In [ ]:
checks = {
    "tmax < tmin": (daily["tmax"] < daily["tmin"]).sum(),
    "tmax > 125 °F": (daily["tmax"] > 125).sum(),
    "tmin < -50 °F": (daily["tmin"] < -50).sum(),
    "prcp < 0": (daily["prcp"] < 0).sum(),
    "prcp > 10 in/day": (daily["prcp"] > 10).sum(),
    "non-integer tmax": (daily["tmax"].dropna() % 1 != 0).sum(),
    "non-0.01 prcp resolution": ((daily["prcp"].dropna() * 100).round(3) % 1 != 0).sum(),
}
pd.Series(checks, name="rows").to_frame()

In [ ]:
# Where do non-integer temperatures come from?
temps = daily.dropna(subset=["tmax"])
frac = temps.assign(decimal=temps["tmax"] % 1 != 0).groupby("Year")["decimal"].mean()
frac[frac > 0]

In [ ]:
daily.loc[daily["tmax"] < daily["tmin"], ["Name", "Date", "tmax", "tmin"]].sort_values("Date").tail(
    10
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(daily["tmax"].dropna(), bins=np.arange(-40, 130, 2), ax=axes[0])
axes[0].set_title("tmax (°F)")
sns.histplot(daily["tmin"].dropna(), bins=np.arange(-70, 100, 2), ax=axes[1])
axes[1].set_title("tmin (°F)")
wet = daily["prcp"].dropna()
sns.histplot(wet[wet > 0], bins=np.logspace(-2, 1.3, 50), ax=axes[2])
axes[2].set_xscale("log")
axes[2].set_title(f"prcp > 0 (log x); {(wet == 0).mean():.0%} of days are exactly 0")
plt.tight_layout()

## 3. Seasonal cycle by region (1989-2018 climatology)

In [ ]:
recent = daily[daily["Year"].between(1989, 2018)]
clim = (
    recent.groupby(["region", "Month"], observed=True)
    .agg(tmax=("tmax", "mean"), tmin=("tmin", "mean"), prcp=("prcp", "mean"))
    .reset_index()
)
clim["prcp_month_in"] = clim["prcp"] * 30.4
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col, title in zip(
    axes,
    ["tmax", "tmin", "prcp_month_in"],
    ["Mean daily tmax (°F)", "Mean daily tmin (°F)", "Mean monthly precip (in)"],
    strict=True,
):
    sns.lineplot(
        data=clim,
        x="Month",
        y=col,
        hue="region",
        palette=REGION_PALETTE,
        marker="o",
        ax=ax,
        legend=ax is axes[-1],
    )
    ax.set_title(title)
    ax.set_xticks(range(1, 13))
axes[-1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()

In [ ]:
# When does the hottest day happen? (how sensitive `hottest_day_temp` is to summer gaps)
hot = recent.dropna(subset=["tmax"])
idx = hot.groupby(["ID", "Year"], observed=True)["tmax"].idxmax()
peak = hot.loc[idx, ["region", "Month"]]
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(
    data=peak,
    x="Month",
    hue="region",
    palette=REGION_PALETTE,
    multiple="stack",
    discrete=True,
    ax=ax,
)
ax.set_title("Month of each city-year's hottest day, 1989-2018")
plt.tight_layout()

## 4. Reconciling daily → annual

Recompute the four target metrics from daily data and compare with `annual_indices`
(`Avg Temp` = mean of (tmax+tmin)/2; `cdd` = Σ max(0, tavg − 65)). If they match, the daily
table can be used to build sub-annual features (e.g. seasonal means) consistent with the targets.

In [ ]:
recent = recent.assign(tavg=(recent["tmax"] + recent["tmin"]) / 2)
recent = recent.assign(cdd_day=(recent["tavg"] - 65).clip(lower=0))
rebuilt = (
    recent.groupby(["ID", "Year"], observed=True)
    .agg(
        avg_temp=("tavg", "mean"),
        tot_prcp=("prcp", "sum"),
        max_tmax=("tmax", "max"),
        cdd_rebuilt=("cdd_day", "sum"),
    )
    .reset_index()
    .astype({"ID": str})
)
cmp = rebuilt.merge(annual, on=["ID", "Year"])
cmp = cmp[cmp[["Miss Tmax", "Miss Tmin", "Miss Prcp"]].max(axis=1) == 0]
diffs = pd.DataFrame(
    {
        "Avg Temp": cmp["avg_temp"] - cmp["Avg Temp"],
        "ToT Prcp": cmp["tot_prcp"] - cmp["ToT Prcp"],
        "Max Tmax": cmp["max_tmax"] - cmp["Max Tmax"],
        "cdd": cmp["cdd_rebuilt"] - cmp["cdd"],
    }
)
print(f"complete city-years compared: {len(cmp):,}")
diffs.abs().describe(percentiles=[0.5, 0.99]).round(3)

## 5. Station switches in the daily record

For cities whose station changed in 1989-2018, compare mean tmax/tmin 3 years before vs after
the switch, against the same difference averaged over all other cities in the same region
(to cancel out regional weather).

In [ ]:
city_info = load_city_info()
switches = city_info.sort_values(["ID", "Stn.stDate"]).groupby("ID").tail(-1)
switches = switches[switches["Stn.stDate"].dt.year.between(1992, 2015)]
monthly = daily.groupby(["ID", "region", pd.Grouper(key="Date", freq="MS")], observed=True)[
    ["tmax", "tmin"]
].mean()
monthly = monthly.reset_index()
monthly["ID"] = monthly["ID"].astype(str)
# anomaly vs regional mean of the same month removes shared weather
monthly[["tmax_anom", "tmin_anom"]] = (
    monthly[["tmax", "tmin"]]
    - monthly.groupby(["region", "Date"], observed=True)[["tmax", "tmin"]]
    .transform("mean")
    .to_numpy()
)

rows = []
for _, sw in switches.iterrows():
    m = monthly[monthly["ID"] == sw["ID"]]
    t0 = sw["Stn.stDate"]
    before = m[(m["Date"] < t0) & (m["Date"] >= t0 - pd.DateOffset(years=3))]
    after = m[(m["Date"] >= t0) & (m["Date"] < t0 + pd.DateOffset(years=3))]
    rows.append(
        {
            "Name": sw["Name"],
            "switch": t0.date(),
            "station": sw["Stn.Name"],
            "tmax_step": after["tmax_anom"].mean() - before["tmax_anom"].mean(),
            "tmin_step": after["tmin_anom"].mean() - before["tmin_anom"].mean(),
        }
    )
steps = pd.DataFrame(rows)
print(f"{len(steps)} switches 1992-2015")
steps.reindex(steps["tmin_step"].abs().sort_values(ascending=False).index).head(12).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    steps.melt(value_vars=["tmax_step", "tmin_step"]),
    x="value",
    hue="variable",
    bins=30,
    element="step",
    ax=ax,
)
ax.axvline(0, color="k", lw=0.8)
ax.set_title("Step change (°F) across station switches, relative to region")
plt.tight_layout()

## Findings: daily table

- **Complete calendar:** every city has one row per day from its first date to 2018-12-31, and
  missing observations are stored as NaN. Missing tmax and tmin almost always occur together.
- **Daily reproduces annual exactly** for complete years: `ToT Prcp`, `Max Tmax` and `cdd` match
  to 0, and `Avg Temp` matches to within 0.01 °F (rounding). `Avg Temp` = mean((tmax+tmin)/2) and
  `cdd` = Σ max(0, tavg − 65 °F). So sub-annual features built from daily data line up with the targets.
- **Precision change in 2018:** temperatures are whole °F up to 2017. In 2018 about 36k values
  have one decimal place. This is minor, but rounding 2018 makes the series consistent.
- **Some cities are mostly missing in the baseline window (1989-2018):** Huntsville is about
  97% missing (from 1960 on), and Akron, Louisville, Wheeling and Trenton are about 30% missing.
  Grand Island has no missing temperatures but is missing about 14% of precipitation.
- **A handful of impossible values:** 9 days with tmax < tmin, all very old except Portland in 1999.
  There are also 12 days with tmin < -50 °F and 55 days with more than 10 in of rain.
  These are plausible extremes, so flag them rather than dropping them.
- **Station switches cause step changes:** in 30 switches from 1992 to 2015, the region-relative
  mean changes by up to about 4 °F (Hayward 1998, Tri-Cities 1998, Denver 1995). This is as large
  as decades of trend, so treat the last switch as a break point.
- **Seasonality differs a lot by region,** and 89% of city-years have their hottest day in
  June–August (July alone is 45%). Missing summer days therefore hurt `hottest_day_temp` and `cdd` much more than
  missing winter days.